# Reference · Day 10 studio — three decisions, one model

**Not a marking key.** Everyone worked on the same split, so if you followed the studio
your numbers should match these — and if they do not, the interesting question is which
choice you made differently.

The studio asked you to write down a threshold *before* computing one. Keep your guesses
next to these numbers as you read: the distance between them is the lesson.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from stat764 import diabetes

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Tuesday's model, exactly as in the Meeting 10 notebook.
enc = diabetes()
enc = enc[~enc["discharge_disposition_id"].isin([11, 13, 14, 19, 20, 21])]  # death, hospice
y_all = (enc["readmitted"] == "<30").astype(int)
NUM = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
       "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
CAT = ["age", "gender", "race", "admission_type_id", "discharge_disposition_id",
       "admission_source_id", "A1Cresult", "max_glu_serum", "insulin", "change",
       "diabetesMed", "medical_specialty"]
X_all = enc[NUM + CAT].astype({c: str for c in CAT})
train, test = next(GroupShuffleSplit(1, test_size=0.25, random_state=764)
                   .split(X_all, y_all, groups=enc["patient_nbr"]))
model = Pipeline([
    ("prep", ColumnTransformer([
        ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT)])),
    ("model", LogisticRegression(max_iter=2000))]).fit(X_all.iloc[train], y_all.iloc[train])
p = model.predict_proba(X_all.iloc[test])[:, 1]
y = y_all.iloc[test].to_numpy()

READMISSION = 17_700            # dollars — AHRQ HCUP Statistical Brief #307 (2024), 2020 data


def net_per_1000(threshold, cost, prevents):
    """Dollars saved minus dollars spent, per 1,000 discharges."""
    flag = p >= threshold
    return ((flag & (y == 1)).sum() * prevents * READMISSION - flag.sum() * cost) / len(y) * 1000


print(f"  {len(y):,} discharges · readmitted {y.mean():.1%} · the same p and y all the way down")

## 1. Screening — a $25 call that prevents 5%

The break-even threshold is a division, and the answer is small: a call is so cheap that
almost any chance of readmission justifies it.

In [ ]:
p_call = 25 / (0.05 * READMISSION)
print(f"  p* = 25 / (0.05 x {READMISSION:,}) = {p_call:.3f}")
print(f"  patients above it: {np.mean(p >= p_call):.1%}   (lowest probability in the data {p.min():.3f})\n")

rows = []
for price in (25, 50, 100, 200, 400):
    ps = price / (0.05 * READMISSION)
    with_model, everyone = net_per_1000(ps, price, 0.05), net_per_1000(0.0, price, 0.05)
    rows.append({"price of a call": f"${price}", "p*": round(ps, 3),
                 "share called": f"{np.mean(p >= ps):.1%}",
                 "using the model": round(with_model), "calling everyone": round(everyone),
                 "what the model adds": round(with_model - max(everyone, 0))})
print(pd.DataFrame(rows).to_string(index=False))

**At \$25 the model is worth nothing.** Nearly every patient clears a 0.028 bar, so "use
the model" and "call everyone" are the same decision — and calling everyone needs no
model at all. As the call gets dearer the bar rises past more patients and the model
starts to earn its keep: a few hundred dollars per 1,000 discharges at \$50, eighteen
thousand at \$100.

⚠ Nothing about the model changed between those rows. Its *value* is a property of the
decision it serves.

⚠ And look at the last row. At \$400 the break-even rule flags a handful of the
highest-risk patients and *loses* money. The rule is only as good as the probabilities:
at the very top this model says more than it delivers — in the Meeting 10 notebook the
top decile said 0.273 and came back at 0.248. That is Tuesday's subject.

## 2. A review queue — 5% of discharges, no more

No price per flag here, only slots. The threshold is the score of the last patient who
fits: the 95th percentile of `p`.

In [ ]:
for capacity in (0.05, 0.10):
    cut = np.quantile(p, 1 - capacity)
    queue = p >= cut
    reached = int((queue & (y == 1)).sum())
    print(f"  {capacity:.0%} queue: threshold {cut:.3f} · reaches {reached} of {y.sum():,} readmissions"
          f" ({reached / y.sum():.1%}) · precision {reached / queue.sum():.3f}"
          f" vs {y.mean():.3f} at random")

The queue is almost three times as productive as reviewing patients at random — and it
still misses most readmissions, because a 5% queue *cannot* reach more than a small share
of an 11% outcome however good the model is. **Capacity sets the ceiling; the model
decides how close you get to it.** Doubling the nurse's time reaches more people at lower
precision — a staffing question, not a modeling one.

⚠ If your precision came out *below* 0.111, check the quantile: `np.quantile(p, 0.05)` is
the bottom 5%.

## 3. An offer that costs real money — a $1,200 program that prevents 25%

In [ ]:
p_prog = 1200 / (0.25 * READMISSION)
cutoffs = np.round(np.arange(0.005, 0.95, 0.001), 3)
f1_cut = cutoffs[int(np.argmax([f1_score(y, p >= c) for c in cutoffs]))]

for rule, c in [("break-even p*", p_prog), ("best F1", f1_cut),
                ("the default, 0.5", 0.5), ("offer everyone", 0.0)]:
    print(f"  {rule:<18} threshold {c:.3f}  flags {np.mean(p >= c):6.1%}"
          f"  net ${net_per_1000(c, 1200, 0.25):>10,.0f} per 1,000")

grid = np.linspace(0.0, 0.6, 241)
plt.figure(figsize=(7, 3.6))
plt.plot(grid, [net_per_1000(c, 1200, 0.25) for c in grid])
plt.axhline(0, color="gray", lw=0.8)
for c, ls in [(p_prog, "--"), (f1_cut, ":")]:
    plt.axvline(c, ls=ls, color="black", lw=1)
plt.ylim(-150_000, 20_000)
plt.xlabel("threshold  (dashed: break-even · dotted: F1)")
plt.ylabel("net $ per 1,000 discharges")
plt.title("$1,200 program, prevents 25%")
plt.tight_layout()
plt.show()

F1's threshold flags more than a quarter of all patients for a \$1,200 program and loses
about a hundred thousand dollars per 1,000 discharges. The default 0.5 makes money — not
because it is a good rule, but because it flags almost no one. **An F1 score contains no
price, so nothing in it could have warned you.**

⚠ Your net value at the break-even threshold depends on which patients landed in the test
set; on other splits it runs from a few thousand dollars to about fourteen thousand per
1,000. Your break-even *threshold* does not move at all — it never looked at a patient.

## The three thresholds, side by side

| decision | what set the threshold | threshold |
|---|---|---|
| $25 call | a price ratio | 0.028 — nearly everyone |
| 5% queue | a count | 0.243 |
| $1,200 program | a price ratio | 0.271 — a few percent |

The model is identical in all three. **Everything that moved was a price, a capacity, or
a promise about what a probability means.**

## What you would say, and what you would refuse to say

| | |
|---|---|
| **"At \$25 a call, call every discharged patient; the model adds nothing at that price."** | ✅ A decision, with the number that makes it. |
| **"We flag patients with a predicted risk above 0.271, the break-even for a \$1,200 program that prevents a quarter of readmissions."** | ✅ Says where the threshold came from — and every number in it can be challenged. |
| **"The model uses a 0.5 threshold."** | ⚠ A default is still a choice. Here it flags almost no one. |
| **"We chose the threshold that maximizes F1."** | ❌ For the program that costs about a hundred thousand dollars per 1,000 discharges, and no F1 score would tell you. |

## Check your own notebook against this

Not "did I get these numbers". These:

| | |
|---|---|
| **1** | Is there a threshold **guess** in each cell, written before the number — and did you keep it when it was wrong? |
| **2** | Does every threshold you report come with **where it came from** — a price ratio, a capacity, or a metric you would have to defend? |
| **3** | If you searched for a best threshold, did you search on patients the threshold is **not** then scored on? |
| **4** | Are your net values **per 1,000 discharges** — the same denominator for every rule — rather than per flagged patient? |
| **5** | Does the notebook survive **Restart & Run All**? |

If 3 is a no, that is the one to fix. A threshold tuned on the patients it is scored on is
Meeting 8's winner's curse in a new place — and it is the easiest one to miss, because a
threshold does not look like a parameter.